## **Imports**

In [1]:
import math
import numpy as np
import torch
import os
from os.path import join as pjoin
import random
import matplotlib
import pandas as pd
import csv

import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.animation import FuncAnimation, FFMpegFileWriter
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
import mpl_toolkits.mplot3d.axes3d as p3

## **Visualize Function**

In [2]:
kinematic_chain = [[0, 2, 5, 8, 11], 
                       [0, 1, 4, 7, 10], 
                       [0, 3, 6, 9, 12, 15], 
                       [9, 14, 17, 19, 21], 
                       [9, 13, 16, 18, 20]]

def plot_3d_motion(save_path, kinematic_tree, mp_joints, title, figsize=(10, 10), fps=120, radius=4):

    title_sp = title.split(' ')
    if len(title_sp) > 20:
        title = '\n'.join([' '.join(title_sp[:10]), ' '.join(title_sp[10:20]), ' '.join(title_sp[20:])])
    elif len(title_sp) > 10:
        title = '\n'.join([' '.join(title_sp[:10]), ' '.join(title_sp[10:])])

    def init():
        ax.set_xlim3d([-radius / 4, radius / 4])
        ax.set_ylim3d([0, radius / 2])
        ax.set_zlim3d([0, radius / 2])
        fig.suptitle(title, fontsize=20)
        ax.grid(b=False)

    def plot_xzPlane(minx, maxx, miny, minz, maxz):
        ## Plot a plane XZ
        verts = [
            [minx, miny, minz],
            [minx, miny, maxz],
            [maxx, miny, maxz],
            [maxx, miny, minz]
        ]
        xz_plane = Poly3DCollection([verts])
        xz_plane.set_facecolor((0.5, 0.5, 0.5, 0.5))
        ax.add_collection3d(xz_plane)

    fig = plt.figure(figsize=figsize)
    ax = fig.add_subplot(111, projection='3d')
    init()

    mp_data = []
    frame_number = min([data.shape[0] for data in mp_joints])

    colors = ['red', 'green', 'black', 'red', 'blue',
              'darkblue', 'darkblue', 'darkblue', 'darkblue', 'darkblue',
              'darkred', 'darkred', 'darkred', 'darkred', 'darkred']

    mp_offset = list(range(-len(mp_joints)//2, len(mp_joints)//2, 1))
    mp_colors = [[colors[i]] * 15 for i in range(len(mp_offset))]

    for i,joints in enumerate(mp_joints):

        # (seq_len, joints_num, 3)
        data = joints.copy().reshape(len(joints), -1, 3)

        MINS = data.min(axis=0).min(axis=0)
        MAXS = data.max(axis=0).max(axis=0)

        height_offset = MINS[1]
        data[:, :, 1] -= height_offset

        mp_data.append({"joints":data,
                        "MINS":MINS,
                        "MAXS":MAXS,})

    def update(index):
        for line in ax.lines:
            line.remove()
        for collection in ax.collections:
            collection.remove()
        ax.view_init(elev=120, azim=-90)
        # plot_xzPlane(-3, 3, 0, -3, 3)
        for pid,data in enumerate(mp_data):
            for i, (chain, color) in enumerate(zip(kinematic_tree, mp_colors[pid])):
                if i < 5:
                    linewidth = 4.0
                else:
                    linewidth = 2.0
                ax.plot3D(data["joints"][index, chain, 0], data["joints"][index, chain, 1], data["joints"][index, chain, 2], linewidth=linewidth,
                          color=color)

        ax.set_axis_off()
        ax.set_xticklabels([])
        ax.set_yticklabels([])
        ax.set_zticklabels([])
        
    ani = FuncAnimation(fig, update, frames=frame_number, interval=1000 / fps, repeat=False)
    ani.save(save_path, fps=fps)
    plt.close()
    
    print('done ! animation saved to {}'.format(save_path))

In [3]:
motion_path = '/home/layesekou/masterds/datachallenge/text-to-motion-retrieval/motions/motions/10.npy'
motion = np.load(motion_path)
print('motion shape:', motion.shape)

motion shape: (94, 384)


In [4]:
## data directory
data_root = '/home/layesekou/masterds/datachallenge/text-to-motion-retrieval'
motion_dir = pjoin(data_root, 'motions', 'motions')
text_dir = pjoin(data_root, 'texts', 'texts')

motion_files = [f for f in os.listdir(motion_dir) if f.endswith('.npy')]
## pick a random file
motion_files = [f for f in os.listdir(pjoin(data_root, 'motions', 'motions')) if f.endswith('.npy')]
fname = random.choice(motion_files)

## load motion & text
motion = np.load(pjoin(motion_dir, fname))
print('motion shape:', motion.shape)
with open(pjoin(text_dir, fname.replace('.npy', '.txt'))) as fr:
    texts = fr.read().strip().split('\n')
    
print('Number of text descriptions:', len(texts))
## pick a random text for visualization
title = random.choice(texts)
print('title:', title)

motion shape: (492, 384)
Number of text descriptions: 3
title: the first one holds the other person's left hand with their right hand and dances together before releasing the hold.


In [5]:
## extract positions for both persons
pos_person_1 = motion[:, :22*3]
pos_person_2 = motion[:, 22*3+21*6:22*3+21*6+22*3]

## save animation
plot_3d_motion('./animation.gif', kinematic_chain, [pos_person_1, pos_person_2], title, radius=4.0, fps=30)

MovieWriter ffmpeg unavailable; using Pillow instead.


done ! animation saved to ./animation.gif


## **Evaluation Function**

In [6]:
def eval_recall(gt_df, pred_df, verbose=False):

    ks = list(range(1, 11))  # Recall@1 ... Recall@9

    rank_cols = list(pred_df.columns)

    ## convert to numpy
    gt = gt_df["candidate"].values
    preds = pred_df[rank_cols].values

    assert len(gt) == len(preds), "size mismatch"
    
    ## compute Recall@K
    recalls = {k: 0 for k in ks}
    n = len(gt)

    for i in range(n):
        for k in ks:
            if gt[i] in preds[i, :k]:
                recalls[k] += 1

    for k in ks:
        recalls[k] /= n

    ## weighted aggregation
    weights = {k: 1.0 / k for k in ks} # higher weight for smaller k
    weight_sum = sum(weights.values())

    final_score = sum(
        weights[k] * recalls[k] for k in ks
    ) / weight_sum
    
    if verbose:
        for k, val in recalls.items():
            print('recall@{} = {}'.format(k, round(val, 3)))

    return final_score

## **Generate Validation Batches**

In [7]:
def generate_val_batches(data_root, save_dir, batch_size, num_batches):
    
    os.makedirs(save_dir, exist_ok=True)

    with open(os.path.join(data_root, 'train.txt')) as fd:
        train_fnames = fd.read().strip().split('\n') 

    random_fnames = random.sample(train_fnames, k=batch_size * num_batches)
    random_batches = np.array(random_fnames).reshape(num_batches, batch_size)

    gt = [['query_id', 'candidate']]
    for file_idx, batch in enumerate(random_batches, start=1):
        os.makedirs(pjoin(save_dir, str(file_idx)), exist_ok=True)

        rdm_idx = random.randint(0, len(batch)-1)
        random_text_fname = batch[rdm_idx]

        with open(os.path.join(data_root, 'texts', 'texts', random_text_fname+'.txt')) as fd:
            file_texts = fd.read().strip().split('\n')
            random_text = random.choice(file_texts)

        with open(os.path.join(save_dir, str(file_idx), 'text.txt'), 'w') as fw:
            fw.write(random_text)

        for motion_idx, fname in enumerate(batch, start=1):
            if fname == random_text_fname:
                gt.append([file_idx, motion_idx])

            motion = np.load(os.path.join(data_root, 'motions', 'motions', fname+'.npy'))

            np.save(pjoin(save_dir, str(file_idx), 'motion_{}.npy'.format(motion_idx)), motion)


    with open(os.path.join(save_dir, 'gt.csv'), 'w', newline='') as file:
        writer = csv.writer(file)
        writer.writerows(gt)
        
    print('Done ! val batches saved at {} !'.format(save_dir))

In [8]:
## data dir
data_root = '/home/layesekou/masterds/datachallenge/text-to-motion-retrieval'

## dir to save batches to
save_dir = '/home/layesekou/masterds/datachallenge/text-to-motion-retrieval/val/'
    
## number of val batches & number of element per batch
batch_size = 32
num_batches = 30

## generate val batches
generate_val_batches(data_root, save_dir, batch_size, num_batches)

Done ! val batches saved at /home/layesekou/masterds/datachallenge/text-to-motion-retrieval/val/ !


### **Evaluate random submission**

In [9]:
## generate random submission (30 queries × 10 candidates)
random_sub = np.array([
    np.random.choice(range(1, 33), size=10, replace=False)
    for _ in range(30)
])

## create DataFrame
random_sub_df = pd.DataFrame(
    random_sub,
    index=np.arange(1, 31),
    columns=[f'candidate_{i}' for i in range(1, 11)]
)

random_sub_df.index.name = 'query_id'
random_sub_df.head()

,candidate_1,candidate_2,candidate_3,candidate_4,candidate_5,candidate_6,candidate_7,candidate_8,candidate_9,candidate_10
query_id,,,,,,,,,,
1,21,1,10,18,31,22,2,9,14,6
2,4,27,5,30,19,1,17,9,22,29
3,18,6,22,12,31,1,24,19,3,10
4,13,17,30,4,25,14,18,7,5,11
5,8,9,14,12,32,26,10,3,24,5


In [10]:
## load gt
gt_df = pd.read_csv(pjoin(save_dir, 'gt.csv'))
gt_df.head()

,query_id,candidate
0,1,24
1,2,11
2,3,27
3,4,14
4,5,14


In [11]:
## evaluate random results
racall = eval_recall(gt_df, random_sub_df, verbose=True)
print('Recall score:', racall)

recall@1 = 0.0
recall@2 = 0.0
recall@3 = 0.033
recall@4 = 0.1
recall@5 = 0.1
recall@6 = 0.133
recall@7 = 0.167
recall@8 = 0.267
recall@9 = 0.333
recall@10 = 0.333
Recall score: 0.07027954658357043


In [12]:
def generate_val_batches_with_split(data_root, save_dir, batch_size, num_batches, val_ratio=0.1, seed=42):
    random.seed(seed)
    np.random.seed(seed)

    os.makedirs(save_dir, exist_ok=True)

    # --- 1. Charger toutes les motions du train
    with open(os.path.join(data_root, 'train.txt')) as fd:
        all_fnames = fd.read().strip().split('\n')

    # --- 2. Filtrer uniquement les motions qui ont un fichier texte
    text_dir = os.path.join(data_root, 'texts', 'texts')
    available_texts = [f.replace('.txt','') for f in os.listdir(text_dir)]
    all_fnames = [f for f in all_fnames if f in available_texts]

    # --- 3. Split train / validation
    random.shuffle(all_fnames)
    split_idx = int((1 - val_ratio) * len(all_fnames))
    train_fnames = all_fnames[:split_idx]
    val_fnames = all_fnames[split_idx:]

    print(f"Total motions: {len(all_fnames)}")
    print(f"Train motions: {len(train_fnames)}")
    print(f"Validation motions: {len(val_fnames)}")

    # --- 4. Sauvegarder le nouveau fichier train_now.txt (sans les motions de validation)
    train_now_path = os.path.join(save_dir, 'train_now.txt')
    with open(train_now_path, 'w') as f:
        for fname in train_fnames:
            f.write(fname + '\n')
    print(f"Saved new train split without validation motions to: {train_now_path}")

    # --- 5. Sauvegarder aussi les IDs de validation pour référence
    val_ids_path = os.path.join(save_dir, 'validation_ids.txt')
    with open(val_ids_path, 'w') as f:
        for fname in val_fnames:
            f.write(fname + '\n')
    print(f"Saved validation IDs to: {val_ids_path}")

    # --- 6. Générer les batches de validation
    if batch_size * num_batches > len(val_fnames):
        raise ValueError("batch_size * num_batches > nombre de motions disponibles pour validation")

    random_fnames = random.sample(val_fnames, k=batch_size * num_batches)
    random_batches = np.array(random_fnames).reshape(num_batches, batch_size)

    gt = [['query_id', 'candidate']]
    mapping = []  # Pour garder trace des correspondances batch_id -> motion_id original
    
    for file_idx, batch in enumerate(random_batches, start=1):
        os.makedirs(pjoin(save_dir, str(file_idx)), exist_ok=True)

        # --- Choisir la motion "correcte" pour le batch
        rdm_idx = random.randint(0, len(batch)-1)
        random_text_fname = batch[rdm_idx]

        # --- Lire une description aléatoire pour la motion
        with open(os.path.join(data_root, 'texts', 'texts', random_text_fname+'.txt')) as fd:
            file_texts = fd.read().strip().split('\n')
            random_text = random.choice(file_texts)

        # --- Sauvegarder la query texte
        with open(os.path.join(save_dir, str(file_idx), 'text.txt'), 'w') as fw:
            fw.write(random_text)

        # --- Copier toutes les motions du batch
        for motion_idx, fname in enumerate(batch, start=1):
            if fname == random_text_fname:
                gt.append([file_idx, motion_idx])
            
            # Sauvegarder le mapping
            mapping.append({
                'batch_id': file_idx,
                'position_in_batch': motion_idx,
                'original_motion_id': fname
            })

            motion = np.load(os.path.join(data_root, 'motions', 'motions', fname+'.npy'))
            np.save(pjoin(save_dir, str(file_idx), f'motion_{motion_idx}.npy'), motion)

    # --- Sauvegarder le fichier ground-truth
    with open(os.path.join(save_dir, 'gt.csv'), 'w', newline='') as file:
        writer = csv.writer(file)
        writer.writerows(gt)

    # --- Sauvegarder le mapping pour pouvoir retrouver les IDs originaux
    mapping_df = pd.DataFrame(mapping)
    mapping_df.to_csv(pjoin(save_dir, 'motion_id_mapping.csv'), index=False)
    print(f"Saved motion ID mapping to: {pjoin(save_dir, 'motion_id_mapping.csv')}")

    print(f'Done ! {num_batches} val batches saved at {save_dir}')

# Exécution
data_root = '/home/layesekou/masterds/datachallenge/text-to-motion-retrieval'
save_dir = '/home/layesekou/masterds/datachallenge/text-to-motion-retrieval/val_split'

batch_size = 32
num_batches = 28
val_ratio = 0.15  # 10% du train pour validation

generate_val_batches_with_split(data_root, save_dir, batch_size, num_batches, val_ratio=val_ratio)

Total motions: 6018
Train motions: 5115
Validation motions: 903
Saved new train split without validation motions to: /home/layesekou/masterds/datachallenge/text-to-motion-retrieval/val_split/train_now.txt
Saved validation IDs to: /home/layesekou/masterds/datachallenge/text-to-motion-retrieval/val_split/validation_ids.txt
Saved motion ID mapping to: /home/layesekou/masterds/datachallenge/text-to-motion-retrieval/val_split/motion_id_mapping.csv
Done ! 28 val batches saved at /home/layesekou/masterds/datachallenge/text-to-motion-retrieval/val_split


In [13]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
from tqdm import tqdm
import os
from os.path import join as pjoin
import math
import random
import pandas as pd
from transformers import AutoModel, AutoTokenizer, get_cosine_schedule_with_warmup

# ==================== CONFIGURATION AMÉLIORÉE ====================
class ImprovedConfig:
    # Data
    data_root = '/home/layesekou/masterds/datachallenge/text-to-motion-retrieval'
    motion_dir = pjoin(data_root, 'motions', 'motions')
    text_dir = pjoin(data_root, 'texts', 'texts')
    val_dir = '/home/layesekou/masterds/datachallenge/text-to-motion-retrieval/val/'
    train_split = "/home/layesekou/masterds/datachallenge/text-to-motion-retrieval/train.txt"

    
    # Motion processing
    motion_dim = 384
    d_model = 512  # Augmenté pour plus de capacité
    num_joints = 44
    pos_dim = 3
    rot_dim = 6
    
    # Text processing (all-MiniLM-L6-v2: 384 dim, meilleur pour similarité sémantique)
    text_model_name = 'sentence-transformers/all-MiniLM-L6-v2'
    text_dim = 384
    max_text_length = 128
    
    # Architecture améliorée
    num_heads = 16  # Plus de têtes d'attention
    num_layers = 6  # Plus de couches
    dropout = 0.2
    max_motion_length = 200
    
    # Training amélioré
    batch_size = 64  # Plus grand batch
    num_epochs = 30
    learning_rate = 1e-4
    weight_decay = 1e-3
    temperature = 0.01  # Température plus basse
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    
    # Optimisation
    gradient_accumulation_steps = 2
    warmup_steps = 1000
    label_smoothing = 0.1
    
    # Augmentations
    augmentation_prob = 0.3
    noise_std = 0.01
    mask_max_len = 20
    shift_max = 10
    
    # Mining (hard negative avec marge)
    hard_negative_k = 5
    mining_start_epoch = 5
    hard_negative_margin = 0.2   # marge pour la loss margin sur les négatifs difficiles
    hard_negative_weight = 0.5  # poids de la loss margin dans la loss totale
    
    # Checkpoint
    save_path = './best_model_improved.pt'
    
config = ImprovedConfig()
print(f"Using device: {config.device}")
print(f"Model dimension: {config.d_model}")
print(f"Batch size: {config.batch_size}")

# ==================== FONCTION D'ÉVALUATION ====================
def eval_recall(gt_df, pred_df, verbose=False):
    ks = list(range(1, 11))
    rank_cols = list(pred_df.columns)
    
    gt = gt_df["candidate"].values
    preds = pred_df[rank_cols].values
    
    assert len(gt) == len(preds), "size mismatch"
    
    recalls = {k: 0 for k in ks}
    n = len(gt)
    
    for i in range(n):
        for k in ks:
            if gt[i] in preds[i, :k]:
                recalls[k] += 1
    
    for k in ks:
        recalls[k] /= n
    
    weights = {k: 1.0 / k for k in ks}
    weight_sum = sum(weights.values())
    
    final_score = sum(weights[k] * recalls[k] for k in ks) / weight_sum
    
    if verbose:
        for k, val in recalls.items():
            print(f'recall@{k} = {round(val, 3)}')
    
    return final_score

# ==================== MOTION ENCODER AMÉLIORÉ ====================
class ImprovedMotionEncoder(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        
        # Projection initiale avec normalisation
        self.input_proj = nn.Sequential(
            nn.Linear(config.motion_dim, config.d_model),
            nn.LayerNorm(config.d_model),
            nn.Dropout(config.dropout)
        )
        
        # Positional encoding apprenable
        self.pos_embedding = nn.Parameter(torch.randn(1, config.max_motion_length, config.d_model))
        
        # Transformer avec attention multi-têtes améliorée
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=config.d_model,
            nhead=config.num_heads,
            dim_feedforward=config.d_model * 4,
            dropout=config.dropout,
            activation='gelu',
            batch_first=True,
            norm_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=config.num_layers)
        
        # Token [CLS] pour l'agrégation
        self.cls_token = nn.Parameter(torch.randn(1, 1, config.d_model))
        
        # Attention pooling: query apprenable pour agréger les frames
        self.attn_query = nn.Parameter(torch.randn(1, 1, config.d_model))
        self.attn_gate = nn.Parameter(torch.tensor(0.5))  # gate entre CLS et attention-pooled
        
        # Projection finale avec gating
        self.output_proj = nn.Sequential(
            nn.Linear(config.d_model, config.d_model * 2),
            nn.GELU(),
            nn.Dropout(config.dropout),
            nn.Linear(config.d_model * 2, config.d_model),
            nn.LayerNorm(config.d_model)
        )
        
    def forward(self, motion, mask=None):
        batch_size = motion.shape[0]
        
        # Projection
        x = self.input_proj(motion)
        
        # Ajout position embedding
        seq_len = x.size(1)
        x = x + self.pos_embedding[:, :seq_len, :]
        
        # Ajout token [CLS]
        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)
        
        # Ajuster le mask
        if mask is not None:
            cls_mask = torch.ones(batch_size, 1, device=mask.device)
            extended_mask = torch.cat([cls_mask, mask], dim=1)
            transformer_mask = ~extended_mask.bool()
        else:
            transformer_mask = None
        
        # Transformer
        x = self.transformer(x, src_key_padding_mask=transformer_mask)
        
        # CLS et séquence (sans CLS)
        cls_out = x[:, 0, :]  # (B, d_model)
        seq_out = x[:, 1:, :]  # (B, L, d_model)
        
        # Attention pooling sur les frames: scores = query @ seq.T, mask, softmax, aggregate
        if mask is not None and seq_out.size(1) > 0:
            attn_scores = (self.attn_query @ seq_out.transpose(1, 2)).squeeze(1)  # (B, L)
            attn_scores = attn_scores.masked_fill(mask == 0, -1e9)
            attn_weights = F.softmax(attn_scores, dim=-1)  # (B, L)
            attn_pooled = torch.bmm(attn_weights.unsqueeze(1), seq_out).squeeze(1)  # (B, d_model)
        else:
            attn_pooled = seq_out.mean(dim=1)
        
        # Combinaison apprenable CLS + attention-pooled
        gate = torch.sigmoid(self.attn_gate)
        pooled = gate * cls_out + (1 - gate) * attn_pooled
        
        # Projection finale
        embedding = self.output_proj(pooled)
        
        return F.normalize(embedding, dim=-1)

# ==================== TEXT ENCODER ====================
class TextEncoder(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        
        # Load pre-trained (all-MiniLM-L6-v2 ou BERT)
        self.bert = AutoModel.from_pretrained(config.text_model_name)
        
        # Fine-tuning partiel: geler les premières couches (compatible BERT / MiniLM)
        for name, param in self.bert.named_parameters():
            if 'encoder.layer.0' in name or 'encoder.layer.1' in name:
                param.requires_grad = False
        
        # Attention pooling: query apprenable sur les tokens (au lieu du seul CLS)
        self.attn_query = nn.Parameter(torch.randn(1, 1, config.text_dim))
        
        # Projection
        self.output_proj = nn.Sequential(
            nn.Linear(config.text_dim, config.d_model * 2),
            nn.GELU(),
            nn.Dropout(config.dropout),
            nn.Linear(config.d_model * 2, config.d_model),
            nn.LayerNorm(config.d_model)
        )
        
    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        hidden = outputs.last_hidden_state  # (B, L, text_dim)
        
        # Attention pooling avec mask (padding = 0)
        attn_scores = (self.attn_query @ hidden.transpose(1, 2)).squeeze(1)  # (B, L)
        attn_scores = attn_scores.masked_fill(attention_mask == 0, -1e9)
        attn_weights = F.softmax(attn_scores, dim=-1)
        pooled = torch.bmm(attn_weights.unsqueeze(1), hidden).squeeze(1)  # (B, text_dim)
        
        embedding = self.output_proj(pooled)
        return F.normalize(embedding, dim=-1)

# ==================== MODÈLE PRINCIPAL AMÉLIORÉ ====================
class ImprovedTMRModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        
        self.motion_encoder = ImprovedMotionEncoder(config)
        self.text_encoder = TextEncoder(config)
        
        # Projections séparées
        self.text_proj = nn.Sequential(
            nn.Linear(config.d_model, config.d_model),
            nn.LayerNorm(config.d_model)
        )
        self.motion_proj = nn.Sequential(
            nn.Linear(config.d_model, config.d_model),
            nn.LayerNorm(config.d_model)
        )
        
        # Température apprenable
        self.logit_scale = nn.Parameter(torch.ones([]) * np.log(1 / config.temperature))
        
    def forward(self, text_input_ids, text_attention_mask, motion, motion_mask=None):
        # Encodage
        text_emb = self.text_encoder(text_input_ids, text_attention_mask)
        motion_emb = self.motion_encoder(motion, motion_mask)
        
        # Projections spécifiques
        text_emb = self.text_proj(text_emb)
        motion_emb = self.motion_proj(motion_emb)
        
        # Normalisation
        text_emb = F.normalize(text_emb, dim=-1)
        motion_emb = F.normalize(motion_emb, dim=-1)
        
        return text_emb, motion_emb
    
    def compute_loss(self, text_emb, motion_emb):
        # Échelle apprenable
        scale = self.logit_scale.exp()
        
        # Matrice de similarité
        logits = scale * torch.matmul(text_emb, motion_emb.T)
        
        # Loss avec label smoothing
        batch_size = text_emb.shape[0]
        labels = torch.arange(batch_size, device=text_emb.device)
        
        loss_t2m = F.cross_entropy(logits, labels, label_smoothing=self.config.label_smoothing)
        loss_m2t = F.cross_entropy(logits.T, labels, label_smoothing=self.config.label_smoothing)
        
        # Calcul de l'accuracy
        with torch.no_grad():
            pred_t2m = logits.argmax(dim=-1)
            acc = (pred_t2m == labels).float().mean()
        
        return (loss_t2m + loss_m2t) / 2, acc
    
    def compute_loss_with_mining(self, text_emb, motion_emb, hard_negatives=None):
        """Loss avec hard negative mining: CE + loss de marge sur les négatifs difficiles."""
        scale = self.logit_scale.exp().clamp(max=100.0)
        logits = scale * torch.matmul(text_emb, motion_emb.T)
        batch_size = text_emb.shape[0]
        labels = torch.arange(batch_size, device=text_emb.device)
        
        # Loss contrastive standard (symétrique)
        loss_t2m = F.cross_entropy(logits, labels, label_smoothing=self.config.label_smoothing)
        loss_m2t = F.cross_entropy(logits.T, labels, label_smoothing=self.config.label_smoothing)
        loss_ce = (loss_t2m + loss_m2t) / 2
        
        # Loss de marge sur les hard negatives: pousser sim(i, neg) < sim(i, pos) - margin
        if hard_negatives is not None and getattr(self.config, 'hard_negative_margin', 0.0) > 0:
            margin = self.config.hard_negative_margin
            weight_hn = getattr(self.config, 'hard_negative_weight', 0.5)
            loss_margin = torch.tensor(0.0, device=text_emb.device)
            sim = torch.matmul(text_emb, motion_emb.T)  # (B, B) non scaled
            for i in range(batch_size):
                pos_sim = sim[i, i]
                for neg_idx in hard_negatives[i]:
                    neg_sim = sim[i, neg_idx]
                    # hinge: max(0, neg_sim - pos_sim + margin)
                    loss_margin = loss_margin + F.relu(neg_sim - pos_sim + margin)
            loss_margin = loss_margin / (batch_size * max(1, len(hard_negatives[0])))
            loss_ce = loss_ce + weight_hn * loss_margin
        
        with torch.no_grad():
            pred_t2m = logits.argmax(dim=-1)
            acc = (pred_t2m == labels).float().mean()
        
        return loss_ce, acc

# ==================== AUGMENTATIONS ====================
class MotionAugmentation:
    def __init__(self, config):
        self.config = config
    
    def time_mask(self, motion):
        """Masquage temporel aléatoire"""
        if random.random() > self.config.augmentation_prob:
            return motion
        
        motion = motion.clone()
        T = motion.shape[0]
        mask_len = random.randint(5, self.config.mask_max_len)
        start = random.randint(0, max(0, T - mask_len))
        motion[start:start+mask_len] = 0
        return motion
    
    def time_shift(self, motion):
        """Décalage temporel"""
        if random.random() > self.config.augmentation_prob:
            return motion
        
        shift = random.randint(-self.config.shift_max, self.config.shift_max)
        if shift > 0:
            motion = torch.cat([torch.zeros(shift, motion.shape[1]), motion[:-shift]])
        elif shift < 0:
            motion = torch.cat([motion[-shift:], torch.zeros(-shift, motion.shape[1])])
        return motion
    
    def gaussian_noise(self, motion):
        """Bruit gaussien"""
        if random.random() > self.config.augmentation_prob:
            return motion
        
        noise = torch.randn_like(motion) * self.config.noise_std
        return motion + noise
    
    def speed_variation(self, motion):
        """Variation de vitesse (interpolation temporelle)"""
        if random.random() > self.config.augmentation_prob:
            return motion
        
        T = motion.shape[0]
        factor = random.uniform(0.8, 1.2)
        new_T = int(T * factor)
        
        # Interpolation linéaire simple
        indices = torch.linspace(0, T-1, new_T)
        indices_floor = indices.long()
        indices_ceil = torch.min(indices_floor + 1, torch.tensor(T-1))
        weights = indices - indices_floor.float()
        
        motion_aug = (1 - weights.unsqueeze(-1)) * motion[indices_floor] + weights.unsqueeze(-1) * motion[indices_ceil]
        
        # Re-pad si nécessaire
        if new_T < T:
            padding = torch.zeros(T - new_T, motion.shape[1])
            motion_aug = torch.cat([motion_aug, padding])
        elif new_T > T:
            motion_aug = motion_aug[:T]
        
        return motion_aug
    
    def __call__(self, motion):
        motion = self.time_mask(motion)
        motion = self.time_shift(motion)
        motion = self.speed_variation(motion)
        motion = self.gaussian_noise(motion)
        return motion

# ==================== HARD NEGATIVE MINING ====================
class HardNegativeMining:
    def __init__(self, model, config):
        self.model = model
        self.config = config
        self.cache = {}
        
    @torch.no_grad()
    def get_hard_negatives(self, text_emb, motion_emb):
        """Trouve les exemples négatifs les plus difficiles"""
        # Matrice de similarité
        sim_matrix = torch.matmul(text_emb, motion_emb.T)
        
        batch_size = sim_matrix.shape[0]
        hard_negatives = []
        
        for i in range(batch_size):
            # Masquer la positive
            sim_matrix[i, i] = -float('inf')
            # Top-k négatifs
            _, indices = torch.topk(sim_matrix[i], self.config.hard_negative_k)
            hard_negatives.append(indices)
            
        return hard_negatives

# ==================== DATASET ====================
class MotionTextDataset(Dataset):
    def __init__(self, config, split='train'):
        self.config = config
        
        # Load split
        with open(config.train_split, 'r') as f:
            all_motions = f.read().strip().split('\n')
        
        # Filter motions that have text files
        available_texts = set([f.replace('.txt', '') for f in os.listdir(config.text_dir)])
        self.motion_ids = [m for m in all_motions if m in available_texts]
        
        # Load tokenizer
        self.tokenizer = AutoTokenizer.from_pretrained(config.text_model_name)
        
        # Pre-load all text descriptions
        self.texts = {}
        for motion_id in self.motion_ids:
            with open(pjoin(config.text_dir, f"{motion_id}.txt"), 'r') as f:
                self.texts[motion_id] = f.read().strip().split('\n')
        
        # Create list of (motion_id, text_idx) pairs
        self.samples = []
        for motion_id in self.motion_ids:
            for text_idx in range(len(self.texts[motion_id])):
                self.samples.append((motion_id, text_idx))
        
        print(f"Dataset {split}: {len(self.samples)} samples from {len(self.motion_ids)} motions")
        
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        motion_id, text_idx = self.samples[idx]
        
        # Load motion
        motion_path = pjoin(self.config.motion_dir, f"{motion_id}.npy")
        motion = np.load(motion_path)
        
        # Truncate or pad motion
        T = motion.shape[0]
        if T > self.config.max_motion_length:
            # Random crop for training
            start = np.random.randint(0, T - self.config.max_motion_length + 1)
            motion = motion[start:start + self.config.max_motion_length]
            mask = torch.ones(self.config.max_motion_length)
        else:
            # Pad
            padding = self.config.max_motion_length - T
            motion = np.pad(motion, ((0, padding), (0, 0)), mode='constant')
            mask = torch.cat([torch.ones(T), torch.zeros(padding)])
        
        # Load text
        text = self.texts[motion_id][text_idx]
        
        # Tokenize text (max_text_length = 128 pour descriptions longues)
        max_len = getattr(self.config, 'max_text_length', 128)
        encoded = self.tokenizer(
            text,
            padding='max_length',
            truncation=True,
            max_length=max_len,
            return_tensors='pt'
        )
        
        return {
            'motion': torch.FloatTensor(motion),
            'motion_mask': mask,
            'input_ids': encoded['input_ids'].squeeze(0),
            'attention_mask': encoded['attention_mask'].squeeze(0),
            'motion_id': motion_id
        }

# ==================== TRAINING LOOP AMÉLIORÉE ====================
def train_epoch_improved(model, dataloader, optimizer, scheduler, config, epoch, miner=None):
    model.train()
    total_loss = 0
    total_acc = 0
    
    # Augmentations
    augmenter = MotionAugmentation(config)
    
    for batch_idx, batch in enumerate(tqdm(dataloader, desc="Training")):
        # Appliquer augmentations
        motion = batch['motion']
        if epoch > 3:  # Commencer les augmentations après quelques époques
            motion = torch.stack([augmenter(m) for m in motion])
        
        motion = motion.to(config.device)
        motion_mask = batch['motion_mask'].to(config.device)
        input_ids = batch['input_ids'].to(config.device)
        attention_mask = batch['attention_mask'].to(config.device)
        
        # Forward
        text_emb, motion_emb = model(input_ids, attention_mask, motion, motion_mask)
        
        # Hard negative mining
        if miner is not None and epoch > config.mining_start_epoch:
            hard_negatives = miner.get_hard_negatives(text_emb, motion_emb)
            loss, acc = model.compute_loss_with_mining(text_emb, motion_emb, hard_negatives)
        else:
            loss, acc = model.compute_loss(text_emb, motion_emb)
        
        # Backward avec gradient accumulation
        loss = loss / config.gradient_accumulation_steps
        loss.backward()
        
        if (batch_idx + 1) % config.gradient_accumulation_steps == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            optimizer.zero_grad()
        
        total_loss += loss.item() * config.gradient_accumulation_steps
        total_acc += acc.item()
        
        # Update learning rate
        if scheduler is not None:
            scheduler.step()
    
    return {
        'loss': total_loss / len(dataloader),
        'acc': total_acc / len(dataloader)
    }

# ==================== VALIDATION ====================
def evaluate_on_val_batches(model, val_dir, config):
    model.eval()
    
    batch_folders = sorted([f for f in os.listdir(val_dir) if f.isdigit()], key=int)
    predictions = []
    tokenizer = AutoTokenizer.from_pretrained(config.text_model_name)
    
    with torch.no_grad():
        for batch_idx in batch_folders:
            batch_path = pjoin(val_dir, batch_idx)
            
            # Load query
            with open(pjoin(batch_path, 'text.txt'), 'r') as f:
                query_text = f.read().strip()
            
            # Tokenize query (même max_length qu'à l'entraînement)
            max_len = getattr(config, 'max_text_length', 128)
            encoded = tokenizer(
                query_text,
                padding='max_length',
                truncation=True,
                max_length=max_len,
                return_tensors='pt'
            )
            input_ids = encoded['input_ids'].to(config.device)
            attention_mask = encoded['attention_mask'].to(config.device)
            
            # Get text embedding
            text_emb = model.text_encoder(input_ids, attention_mask)
            text_emb = model.text_proj(text_emb)
            text_emb = F.normalize(text_emb, dim=-1)
            
            # Load all motions in batch
            motion_scores = []
            motion_ids = []
            
            for i in range(1, 33):
                motion_path = pjoin(batch_path, f'motion_{i}.npy')
                motion = np.load(motion_path)
                
                # Process motion
                T = motion.shape[0]
                if T > config.max_motion_length:
                    start = (T - config.max_motion_length) // 2
                    motion = motion[start:start + config.max_motion_length]
                    mask = torch.ones(config.max_motion_length)
                else:
                    padding = config.max_motion_length - T
                    motion = np.pad(motion, ((0, padding), (0, 0)), mode='constant')
                    mask = torch.cat([torch.ones(T), torch.zeros(padding)])
                
                motion_tensor = torch.FloatTensor(motion).unsqueeze(0).to(config.device)
                mask_tensor = mask.unsqueeze(0).to(config.device)
                
                # Get motion embedding
                motion_emb = model.motion_encoder(motion_tensor, mask_tensor)
                motion_emb = model.motion_proj(motion_emb)
                motion_emb = F.normalize(motion_emb, dim=-1)
                
                # Compute similarity
                similarity = torch.matmul(text_emb, motion_emb.T).item()
                motion_scores.append(similarity)
                motion_ids.append(i)
            
            # Rank motions
            ranked_indices = np.argsort(motion_scores)[::-1][:10]
            ranked_motion_ids = [motion_ids[idx] for idx in ranked_indices]
            predictions.append([int(batch_idx)] + ranked_motion_ids)
    
    # Create submission DataFrame
    pred_df = pd.DataFrame(
        predictions,
        columns=['query_id'] + [f'candidate_{i}' for i in range(1, 11)]
    ).set_index('query_id')
    
    return pred_df

Using device: cuda
Model dimension: 512
Batch size: 64


In [14]:
# ==================== MAIN TRAINING SCRIPT AMÉLIORÉ ====================
def main():
    # Create datasets
    dataset = MotionTextDataset(config, split='train')
    
    # Split into train/val
    train_size = int(1 * len(dataset))
    val_size = len(dataset) - train_size
    train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])
    
    # Create dataloaders
    train_loader = DataLoader(
        train_dataset, 
        batch_size=config.batch_size,
        shuffle=True,
        num_workers=4,
        pin_memory=True,
        drop_last=True
    )
    
    # Create model
    model = ImprovedTMRModel(config).to(config.device)
    
    # Count parameters
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total parameters: {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")
    
    # Optimizer avec param groups
    optimizer = torch.optim.AdamW([
        {'params': model.motion_encoder.parameters(), 'lr': config.learning_rate},
        {'params': model.text_encoder.parameters(), 'lr': config.learning_rate * 0.1},  # Plus faible pour BERT
        {'params': model.text_proj.parameters(), 'lr': config.learning_rate},
        {'params': model.motion_proj.parameters(), 'lr': config.learning_rate},
        {'params': model.logit_scale, 'lr': config.learning_rate}
    ], weight_decay=config.weight_decay)
    
    # Scheduler avec warmup
    num_training_steps = len(train_loader) * config.num_epochs // config.gradient_accumulation_steps
    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=config.warmup_steps,
        num_training_steps=num_training_steps
    )
    
    # Hard negative miner
    miner = HardNegativeMining(model, config)
    
    # Training loop
    best_recall = 0
    patience = 10
    patience_counter = 0
    
    for epoch in range(config.num_epochs):
        print(f"\nEpoch {epoch + 1}/{config.num_epochs}")
        print(f"Learning rate: {optimizer.param_groups[0]['lr']:.2e}")
        
        # Train
        train_metrics = train_epoch_improved(model, train_loader, optimizer, scheduler, config, epoch, miner)
        print(f"Train - Loss: {train_metrics['loss']:.4f}, Acc: {train_metrics['acc']:.4f}")
        
        # Validate
        pred_df = evaluate_on_val_batches(model, config.val_dir, config)
        gt_df = pd.read_csv(pjoin(config.val_dir, 'gt.csv'))
        recall_score = eval_recall(gt_df, pred_df, verbose=True)
        print(f"Validation Recall Score: {recall_score:.4f}")
        
        # Save best model
        if recall_score > best_recall:
            best_recall = recall_score
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_recall': best_recall,
            }, config.save_path)
            print(f"New best model saved with recall: {best_recall:.4f}")
            patience_counter = 0
        else:
            patience_counter += 1
            
        # Early stopping
        if patience_counter >= patience:
            print(f"Early stopping after {epoch + 1} epochs")
            break
    
    print(f"\nTraining completed. Best Recall: {best_recall:.4f}")
    
    # Load best model for final evaluation
    checkpoint = torch.load(config.save_path)
    model.load_state_dict(checkpoint['model_state_dict'])
    
    # Final evaluation
    pred_df = evaluate_on_val_batches(model, config.val_dir, config)
    gt_df = pd.read_csv(pjoin(config.val_dir, 'gt.csv'))
    final_score = eval_recall(gt_df, pred_df, verbose=True)
    print(f"\nFinal Recall Score with best model: {final_score:.4f}")

if __name__ == "__main__":
    main()

Dataset train: 18054 samples from 6018 motions


/home/layesekou/masterds/.venv/lib/python3.12/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(
2026-02-19 15:39:05.564954: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-19 15:39:06.447898: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-19 15:39:09.949280: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different

Total parameters: 44,428,034
Trainable parameters: 40,879,106

Epoch 1/30
Learning rate: 0.00e+00


Training:   0%|          | 0/282 [00:00<?, ?it/s]/home/layesekou/masterds/.venv/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:192: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn(
Training: 100%|██████████| 282/282 [02:26<00:00,  1.92it/s]


Train - Loss: 5.1578, Acc: 0.0217
recall@1 = 0.133
recall@2 = 0.233
recall@3 = 0.3
recall@4 = 0.367
recall@5 = 0.433
recall@6 = 0.467
recall@7 = 0.5
recall@8 = 0.533
recall@9 = 0.6
recall@10 = 0.633
Validation Recall Score: 0.2985
New best model saved with recall: 0.2985

Epoch 2/30
Learning rate: 2.82e-05


Training:   0%|          | 0/282 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALL

... (35 outputs left out)

Training:   0%|          | 0/282 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALL

Train - Loss: 1.8424, Acc: 0.6653
recall@1 = 0.8
recall@2 = 1.0
recall@3 = 1.0
recall@4 = 1.0
recall@5 = 1.0
recall@6 = 1.0
recall@7 = 1.0
recall@8 = 1.0
recall@9 = 1.0
recall@10 = 1.0
Validation Recall Score: 0.9317

Epoch 21/30
Learning rate: 4.01e-05


Training:   0%|          | 0/282 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALL

KeyboardInterrupt: 

In [15]:
# Load best model, run inference on test set and save submission CSV
test_dir = pjoin(data_root, 'test', 'test')
model = ImprovedTMRModel(config).to(config.device)
config.save_path = '/home/layesekou/masterds/datachallenge/text-to-motion-retrieval/best_model_improved.pt' 

# Load checkpoint with weights_only=False to handle custom Config class
state = torch.load(config.save_path, map_location=config.device, weights_only=False)
if isinstance(state, dict) and 'model_state_dict' in state:
    model.load_state_dict(state['model_state_dict'])
else:
    model.load_state_dict(state)

model.eval()
pred_df = evaluate_on_val_batches(model, test_dir, config)

# Ensure predictions are in the correct format for submission
submission_df = pred_df.reset_index()
submission_df.columns = ['query_id'] + [f'candidate_{i}' for i in range(1, 11)]

# Verify the format
print("Submission format:")
print(submission_df.head())
print(f"\nShape: {submission_df.shape}")
print(f"Columns: {submission_df.columns.tolist()}")

# Save to CSV
submission_path = pjoin('/home/layesekou/masterds/datachallenge/text-to-motion-retrieval/submission_now_3.csv')
submission_df.to_csv(submission_path, index=False)
print(f"\nSubmission saved to: {submission_path}")

# Verify saved file
verify_df = pd.read_csv(submission_path)
print("\nVerification - First 3 rows of saved submission:")
print(verify_df.head(3))


/home/layesekou/masterds/.venv/lib/python3.12/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


Submission format:
   query_id  candidate_1  candidate_2  candidate_3  candidate_4  candidate_5  \
0         1           30           17           15            5           10   
1         2           11            7           31           24           28   
2         3            4           28           30           22           18   
3         4           31            7           20           24            1   
4         5           28           24           13            3           17   

   candidate_6  candidate_7  candidate_8  candidate_9  candidate_10  
0           26            1           27           29             9  
1            8           10           16           18            21  
2           29           21            9           16             7  
3           29            6           26           28            14  
4           10            4            8           18            12  

Shape: (30, 11)
Columns: ['query_id', 'candidate_1', 'candidate_2', 'candidate_